# Tutorial 02 -- Foundation modes A, B, C, D

This notebook walks through each of the four Op³ foundation modes with a worked example, showing how the first natural frequency responds to progressively more faithful soil representations. The test subject is the NREL 5 MW OC3 monopile.

## Mode A: Fixed base

In [ ]:
from op3 import build_foundation, compose_tower_model

model_A = compose_tower_model(
    rotor='nrel_5mw_baseline',
    tower='nrel_5mw_oc3_tower',
    foundation=build_foundation(mode='fixed'),
)
f1_A = model_A.eigen(n_modes=3)[0]
print(f'Mode A (fixed): f1 = {f1_A:.4f} Hz')

## Mode B: 6x6 lumped stiffness from PISA

In [ ]:
from op3.foundations import foundation_from_pisa
from op3.standards.pisa import SoilState

profile = [
    SoilState(0.0,  5.0e7, 35, 'sand'),
    SoilState(15.0, 1.0e8, 35, 'sand'),
    SoilState(36.0, 1.5e8, 36, 'sand'),
]
fnd_B = foundation_from_pisa(diameter_m=6.0, embed_length_m=36.0, soil_profile=profile)
model_B = compose_tower_model(rotor='nrel_5mw_baseline', tower='nrel_5mw_oc3_tower', foundation=fnd_B)
f1_B = model_B.eigen(n_modes=3)[0]
print(f'Mode B (PISA 6x6): f1 = {f1_B:.4f} Hz')

## Mode C: Distributed BNWF

In [ ]:
import numpy as np, pandas as pd, tempfile, os
with tempfile.NamedTemporaryFile(mode='w', suffix='.csv', delete=False) as f:
    df = pd.DataFrame({'depth_m': np.linspace(0, 36, 19),
                       'k_ini_kN_per_m': np.linspace(8e4, 2.5e5, 19),
                       'p_ult_kN_per_m': np.linspace(2e3, 1.2e4, 19),
                       'spring_type': ['lateral']*19})
    df.to_csv(f.name, index=False)
    csv_path = f.name
model_C = compose_tower_model(
    rotor='nrel_5mw_baseline', tower='nrel_5mw_oc3_tower',
    foundation=build_foundation(mode='distributed_bnwf', spring_profile=csv_path))
f1_C = model_C.eigen(n_modes=3)[0]
print(f'Mode C (distributed BNWF): f1 = {f1_C:.4f} Hz')
os.unlink(csv_path)

## Mode D: Dissipation-weighted BNWF

In [ ]:
with tempfile.NamedTemporaryFile(mode='w', suffix='.csv', delete=False) as f_sp, \
     tempfile.NamedTemporaryFile(mode='w', suffix='.csv', delete=False) as f_ds:
    pd.DataFrame({'depth_m': np.linspace(0, 36, 19),
                  'k_ini_kN_per_m': np.linspace(8e4, 2.5e5, 19),
                  'p_ult_kN_per_m': np.linspace(2e3, 1.2e4, 19),
                  'spring_type': ['lateral']*19}).to_csv(f_sp.name, index=False)
    D = 100.0 * np.exp(-((np.linspace(0, 36, 19) - 6.0) / 5.0) ** 2)
    pd.DataFrame({'depth_m': np.linspace(0, 36, 19), 'D_total_kJ': D}).to_csv(f_ds.name, index=False)
    sp_path, ds_path = f_sp.name, f_ds.name

model_D = compose_tower_model(
    rotor='nrel_5mw_baseline', tower='nrel_5mw_oc3_tower',
    foundation=build_foundation(mode='dissipation_weighted',
                                 spring_profile=sp_path, ogx_dissipation=ds_path,
                                 mode_d_alpha=2.0, mode_d_beta=0.05))
f1_D = model_D.eigen(n_modes=3)[0]
print(f'Mode D (alpha=2, beta=0.05): f1 = {f1_D:.4f} Hz')
print(f'\nMonotone flex ranking: A > B >= C >= D')
print(f'  A: {f1_A:.4f}  B: {f1_B:.4f}  C: {f1_C:.4f}  D: {f1_D:.4f}')
os.unlink(sp_path); os.unlink(ds_path)